# DSC@UIT 2026 — Task 1: LegalIR
**Team: SGU_LegalMind** · Codabench: https://www.codabench.org/competitions/17715/

Pipeline truy vấn văn bản pháp luật tiếng Việt:

```
question ─┬─► BM25 (chunk)   ─┐
          └─► dense (chunk)  ─┴─► RRF fuse ─► top-N doc ─► cross-encoder rerank ─► top-5
```

## Layout Drive thực tế

```
[Shared with me] UIT DS Challenge/        ← owner: phatleh.224@gmail.com
└── Task 1/
    ├── LegalR - Public Test/             ← DỮ LIỆU (notebook tự dò tới đây)
    │     train.json · public-official.json · selected-contexts.zip
    ├── Scoring-Program-Task-LegalIR/     ← scoring.py chính thức của BTC
    ├── SourceCode/                       ← notebook này
    ├── Vòng 1 - Warm up/
    └── Vòng 2 - PublicTest/

[My Drive] DSC2026_Task1/artifacts/       ← notebook GHI vào đây (~1.5GB)
```

Artifact ghi vào **My Drive của bạn**, không ghi vào thư mục chia sẻ của nhóm — vừa
tránh làm rối thư mục chung, vừa tránh bị người khác xoá mất 1.5GB kết quả encode.

## ⚠ Bắt buộc làm 1 lần trước khi chạy

`drive.mount()` **chỉ mount My Drive**. Thư mục `UIT DS Challenge` đang ở
**"Shared with me"** nên Colab *không* thấy nó → phải tạo shortcut:

> drive.google.com → **Shared with me** → chuột phải **UIT DS Challenge**
> → **Organise** → **Add shortcut to Drive** → chọn **My Drive**

Sau bước này notebook **tự dò** ra đường dẫn, không cần sửa gì. Nếu vẫn
không ra, cell "Tự dò thư mục dữ liệu" sẽ in hướng dẫn và bạn gán tay biến `RAW`.

## Cách dùng
1. Làm bước shortcut ở trên.
2. Chạy tuần tự các Stage. Mỗi stage **cache artifact ra Drive** và tự bỏ qua nếu đã có
   → session Colab chết giữa đường thì chạy lại không mất việc đã làm.
3. Chạy trên **toàn bộ dữ liệu thật** (8.532 văn bản). Bước tốn thời gian duy nhất
   là Stage 5 encode corpus (~20 phút trên Kaggle 2×T4), và nó cache lại nên chỉ
   phải trả giá một lần.

## Mốc so sánh
Baseline TF-IDF toàn văn bản (truncate 20k ký tự), đo trên 500 câu train:

| | recall@5 | recall@20 | recall@100 |
|---|---|---|---|
| TF-IDF doc-level | **0.486** | 0.698 | 0.870 |

Khoảng cách `@100 = 0.87` vs `@5 = 0.486` cho thấy phần lớn điểm mất nằm ở **ranking**,
không phải retrieval → reranker là bước ăn điểm mạnh nhất.

## Ràng buộc cuộc thi (đọc từ `scoring.py` chính thức, không phải từ mô tả)
- **Tối đa 5 document_id / câu hỏi.** Vượt quá ở bất kỳ câu nào → câu đó nhận **0** cả
  Recall lẫn Precision.
- **Recall không cắt top-5** — code BTC chỉ *chặn* ở `len <= 5`, rồi lấy giao của
  **toàn bộ** danh sách nộp. Nên nộp **đúng 5 id** là tối ưu cho Recall.
- **Lệch số câu hoặc thiếu qid → `raise Exception` / `TypeError` → sập TOÀN BỘ bài nộp**,
  không phải chỉ mất điểm câu đó. Validator ở Stage 9 chặn cả hai.
- **id phải là string.** Code chấm dùng `set(truth) & set(pred)`; nộp `123` (int) sẽ
  không khớp `"123"` (str) → 0 điểm dù dự đoán đúng.
- **Recall là metric chính**, Precision chỉ dùng tie-break → **luôn nộp đủ 5 id**.
- **Tổng tham số toàn hệ thống < 4 tỷ** (gồm cả embedding layer).
- **Cấm mọi API.** Chỉ model open-source tự host.
- **Phải đăng ký model với BTC trước khi dùng** — xem cuối notebook.

## Model dùng trong notebook này

| Vai | Model | Tham số | Context |
|---|---|---|---|
| bi-encoder | `bkai-foundation-models/vietnamese-bi-encoder` | 135M | 256 |
| reranker | `BAAI/bge-reranker-v2-m3` | 568M | 512 |
| | **Tổng** | **0.70B** < 4B | |

---

## Chạy ở đâu: Colab · Kaggle (2 GPU) · local

Notebook tự nhận môi trường ở Stage 0 (`ENV`) rồi tự chọn đường dẫn — không phải sửa gì.

| | Colab | Kaggle | local |
|---|---|---|---|
| đọc dữ liệu | My Drive (cần shortcut) | `/kaggle/input/<dataset>` (Add Input) | thư mục hiện tại |
| ghi artifact | `MyDrive/DSC2026_Task1/artifacts` | `/kaggle/working/DSC2026_Task1/artifacts` | `./work/artifacts` |
| đĩa tạm | `/content/scratch` | `/kaggle/temp/scratch` | `./scratch` |
| GPU | 1×T4 16GB | **2×T4 15GB** (hoặc 1×P100) | tuỳ máy |

### Kaggle: 3 việc phải làm

1. **Add Input** → Datasets → hai dataset đã ghim sẵn đường dẫn trong Stage 0:

   ```
   uit-task1-publictest  ->  .../uit-task1-publictest/LegalR - Public Test
   uit-task1-scoring     ->  .../uit-task1-scoring/Scoring-Program-Task-LegalIR
   ```

   Cả hai dạng `/kaggle/input/<slug>/...` và `/kaggle/input/datasets/<user>/<slug>/...`
   đều được nhận. Đổi tên dataset hay đổi tài khoản thì sửa `KAGGLE_DATA_HINTS` /
   `KAGGLE_SCORING_HINTS` ở Stage 0 — hoặc kệ, không trúng thì notebook tự dò
   trong `/kaggle/input`.

   Dataset scoring không bắt buộc, nhưng có thì Stage 3 sẽ đối chiếu hàm chấm điểm
   tự viết với `scoring.py` thật của BTC.
2. **Settings → Accelerator → GPU T4 ×2** (mặc định là None). Có 2 GPU thì Stage 5 và
   Stage 7 tự chia việc; 1 GPU vẫn chạy đúng, chỉ chậm hơn.
3. **Settings → Internet → On** — cần để tải trọng số model từ HuggingFace.

### Kaggle: giữ lại embedding giữa các session

Kaggle không có Drive, session chết là mất hết ngoài `/kaggle/working`. Vòng lặp đúng:

> chạy Stage 1–5 → **Save Version** (Save & Run All hoặc Quick Save) → lần sau
> **Add Input → Your Work / Notebook Output** chính notebook đó

Stage 0 tự quét mọi `artifacts/` trong `/kaggle/input` (biến `ART_IN`) và Stage 5 đọc
lại từng shard embedding đã có ở đó → chỉ encode phần còn thiếu. Xem Stage 11 ở cuối.

Giới hạn cần biết: `/kaggle/working` **20GB**, session GPU **9h/tuần**, tối đa **12h/lần**.

### Cái gì thực sự chạy trên 2 GPU

| Stage | Cách dùng 2 GPU | Ghi chú |
|---|---|---|
| 5 · encode corpus | `start_multi_process_pool` — 1 process/GPU | bước nặng nhất, gần ×2 |
| 7 · rerank | 1 CrossEncoder/GPU, chia query theo thread | gần ×2 |
| 6 · dense search | không — 1 matmul, để trọn trên `cuda:0` | chia chỉ thêm đồng bộ |
| 8 · fine-tune | **nên chạy 1 GPU** | DataParallel cắt in-batch negatives của MNRL |


---
## Stage 0 — Config & môi trường

In [ ]:
#@title Config
import os, sys, glob

# ---- Môi trường: Colab / Kaggle / local -------------------------------------
# Kiểm tra Kaggle TRƯỚC vì Kaggle không có /content, còn Colab thì luôn có.
# Trên Windows '/content' được hiểu là 'C:\content' nên phải chặn bằng os.name.
def _detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or os.environ.get('KAGGLE_URL_BASE'):
        return 'kaggle'
    if os.name == 'posix' and os.path.isdir('/kaggle/input'):
        return 'kaggle'
    if 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG'):
        return 'colab'
    if os.name == 'posix' and os.path.isdir('/content'):
        return 'colab'
    return 'local'

ENV       = _detect_env()
ON_COLAB  = ENV == 'colab'
ON_KAGGLE = ENV == 'kaggle'
PROFILE   = 'v1'

# ---- GPU ---------------------------------------------------------------------
# '' = dùng hết GPU thấy được (Kaggle: 2×T4 -> Stage 5 và 7 chia việc cho cả hai).
# '0' = buộc 1 GPU. CẦN cho Stage 8 fine-tune: MultipleNegativesRankingLoss phụ
# thuộc in-batch negatives, mà DataParallel cắt batch ra 2 nửa rồi tính loss
# riêng từng nửa -> số negative mỗi mẫu giảm một nửa, train yếu đi.
# Đổi biến này BẮT BUỘC phải Restart runtime rồi chạy lại từ đầu (torch đọc
# CUDA_VISIBLE_DEVICES một lần duy nhất lúc khởi tạo).
VISIBLE_GPUS = ''   #@param ['', '0', '1', '0,1'] {allow-input: true}
if VISIBLE_GPUS:
    os.environ['CUDA_VISIBLE_DEVICES'] = VISIBLE_GPUS

# ---- Nơi GHI artifact --------------------------------------------------------
# Colab : My Drive CỦA BẠN (không ghi vào thư mục chia sẻ của nhóm: ~1.5GB và
#         người khác có thể xoá mất).
# Kaggle: /kaggle/working — 20GB, và là thư mục DUY NHẤT còn lại sau "Save Version".
#         /kaggle/temp không vào output nên dùng làm đĩa tạm.
if ON_COLAB:
    ART_ROOT = '/content/drive/MyDrive/DSC2026_Task1'
    SCRATCH  = '/content/scratch'
elif ON_KAGGLE:
    ART_ROOT = '/kaggle/working/DSC2026_Task1'
    SCRATCH  = '/kaggle/temp/scratch'
else:
    ART_ROOT = './work'
    SCRATCH  = './scratch'
ART = f'{ART_ROOT}/artifacts'

# ---- Cache CHỈ ĐỌC từ session trước (Kaggle) ---------------------------------
# Kaggle không có Drive: cách tái dùng embedding giữa các session là "Save Version"
# rồi attach output đó lại làm Dataset ở lần sau (xem Stage 11 ở cuối notebook).
# ART_IN là các thư mục artifacts tìm thấy trong /kaggle/input.
ART_IN = []
if ON_KAGGLE:
    for pat in ('/kaggle/input/*/artifacts',
                '/kaggle/input/*/DSC2026_Task1/artifacts',
                '/kaggle/input/*/*/artifacts'):
        ART_IN += [p for p in sorted(glob.glob(pat)) if os.path.isdir(p)]
    ART_IN = list(dict.fromkeys(ART_IN))

def cache_path(rel):
    """Tìm cache theo thứ tự: ART (ghi được) -> ART_IN (chỉ đọc). None nếu chưa có."""
    p = f'{ART}/{rel}'
    if os.path.exists(p):
        return p
    for d in ART_IN:
        q = f'{d}/{rel}'
        if os.path.exists(q):
            return q
    return None

def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

# ---- Dataset đã attach trên Kaggle -------------------------------------------
# Kaggle mount dataset theo /kaggle/input/<slug>, nhưng khi attach kèm tên chủ sở
# hữu thì đường dẫn thành /kaggle/input/datasets/<user>/<slug>. Ghim cả hai dạng;
# không trúng dạng nào thì cell sau tự dò trong /kaggle/input.
KAGGLE_DATA_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test',
    '/kaggle/input/uit-task1-publictest/LegalR - Public Test',
]
KAGGLE_SCORING_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR',
    '/kaggle/input/uit-task1-scoring/Scoring-Program-Task-LegalIR',
]
# scoring.py THẬT của BTC -> Stage 3 dùng để đối chiếu hàm chấm điểm tự viết
SCORING_DIR = first_existing(KAGGLE_SCORING_HINTS + [
    './Scoring-Program-Task-LegalIR', '../Scoring-Program-Task-LegalIR'])

# ---- Nơi ĐỌC dữ liệu --------------------------------------------------------
# Kaggle: lấy đường dẫn ghim ở KAGGLE_DATA_HINTS. Colab: cell sau tự dò trong
# My Drive (thư mục chia sẻ ở "Shared with me" không được drive.mount() mount).
DATA_FILES = ('train.json', 'public-official.json')
RAW = first_existing(KAGGLE_DATA_HINTS) if ON_KAGGLE else None

# ---- Chunking ----------------------------------------------------------------
CHUNK_CHARS        = 1000   # khớp MAX_SEQ của bi-encoder (~4 ký tự/token)
CHUNK_OVERLAP      = 200
MAX_CHUNKS_PER_DOC = 300    # chặn văn bản khổng lồ (max corpus: 6M ký tự!)

# ---- Retrieval ---------------------------------------------------------------
BM25_K1, BM25_B = 0.9, 0.4  # tuned cho document dài
CAND_CHUNKS     = 1000      # số chunk lấy ra trước khi gộp về document
CAND_DOCS       = 100       # số doc mỗi nhánh đưa vào RRF
RRF_K           = 60
RERANK_TOP      = 50       # số doc đưa vào cross-encoder
TOPK_SUBMIT     = 5         # RÀNG BUỘC CỨNG của BTC

# Cách dùng điểm của cross-encoder. ĐỪNG đoán — Stage 7 in cả ba, chọn theo val:
#   'rrf'     : RRF giữa thứ tự hybrid và thứ tự rerank. Reranker sai một câu thì
#               phiếu của retriever vẫn giữ được gold trong top-5 -> an toàn nhất.
#   'replace' : tin hẳn reranker (cách cũ). Chỉ chọn khi val cho thấy nó thắng rõ.
#   'none'    : bỏ rerank. Đúng là một lựa chọn: ở lần chạy thử trên corpus rút
#               gọn, hybrid RRF (recall@5 0.919) còn cao hơn 'replace' (0.858).
RERANK_MODE     = 'rrf'   #@param ['rrf', 'replace', 'none']

# ---- Models (phải đăng ký với BTC trước khi dùng) ----------------------------
BIENCODER = 'bkai-foundation-models/vietnamese-bi-encoder'   # 135M
RERANKER  = 'BAAI/bge-reranker-v2-m3'    # 568M   -> tổng 0.70B < 4B
MAX_SEQ   = 256       # token/chunk cho bi-encoder
FT_BIENCODER_PATH = f'{ART}/models/biencoder-ft-{PROFILE}'   # Stage 8 ghi ra đây
# Stage 5 encode bằng checkpoint nào lúc BẮT ĐẦU chạy notebook.
#   False -> checkpoint gốc. Đây là giá trị đúng cho lần chạy đầu: Stage 8 cần
#            retrieval của model gốc để mine hard negative.
#   Stage 8 train xong sẽ TỰ bật biến này lên True + encode lại toàn corpus,
#   nên không cần sửa tay rồi chạy lại notebook từ đầu nữa.
USE_FINETUNED = False
DO_FINETUNE   = True    #@param {type:'boolean'} — Stage 8 có train bi-encoder không

# ---- Eval --------------------------------------------------------------------
N_VAL = 1000
SEED  = 42
FT_BATCH_X2_ON_MULTIGPU = True   # xem giải thích ở cell Stage 8

# ---- Batch: các số này là batch cho MỖI GPU ---------------------------------
# T4 Colab 16GB và T4 Kaggle 15GB gần như bằng nhau -> giữ nguyên batch, tốc độ
# tăng là do chia việc cho 2 GPU chứ không phải do batch to hơn. Kaggle P100
# (1×16GB) cũng chạy đúng cấu hình này.
ENC_SHARD     = 50000   # encode theo shard, checkpoint từng shard ra đĩa
ENC_BATCH     = 128
RERANK_MAXLEN = 512
RERANK_BATCH  = 32
RERANK_FP16   = True     # reranker chạy fp16: ~2x nhanh, chênh lệch điểm không đáng kể

print('ENV =', ENV, '| PROFILE =', PROFILE, '| RERANK_MODE =', RERANK_MODE)
print('ART      =', ART)
print('SCRATCH  =', SCRATCH)
if RAW:
    print('RAW      =', RAW)
if SCORING_DIR:
    print('scoring.py BTC:', SCORING_DIR)
if ART_IN:
    print('cache chỉ đọc:', ART_IN)


In [ ]:
#@title Mount Drive (Colab) + cài dependency + liệt kê GPU
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Colab: pin đúng bản đã test. Kaggle: image đã có sentence-transformers sẵn và
# việc downgrade hay kéo theo xung đột transformers/torch -> chỉ cài khi thiếu.
ST_WANT = '3.0.1'
try:
    import sentence_transformers as _st
    ST_HAVE = _st.__version__
except ImportError:
    ST_HAVE = None

if ST_HAVE is None or (ON_COLAB and ST_HAVE != ST_WANT):
    !pip -q install sentence-transformers=={ST_WANT} pyarrow 2>&1 | tail -1
    import importlib, sentence_transformers as _st
    ST_HAVE = importlib.reload(_st).__version__

print('sentence-transformers', ST_HAVE, '(bản đã test:', ST_WANT + ')')

import os
for d in (ART, f'{ART}/emb', f'{ART}/models', SCRATCH):
    os.makedirs(d, exist_ok=True)

import torch
N_GPU   = torch.cuda.device_count()
DEVICES = [f'cuda:{i}' for i in range(N_GPU)] or ['cpu']
DEV     = DEVICES[0]

print('torch', torch.__version__, '| GPU thấy được:', N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB')
if N_GPU > 1:
    print(f'-> Stage 5 (encode corpus) và Stage 7 (rerank) sẽ chia việc cho {N_GPU} GPU')
    print('   Stage 8 (fine-tune) nên chạy 1 GPU: đặt VISIBLE_GPUS="0" + Restart runtime.')
elif N_GPU == 1:
    print('-> 1 GPU: pipeline chạy tuần tự như bình thường')
else:
    print('-> KHÔNG có GPU. Stage 1-4 (BM25) vẫn chạy được và nên chạy ở session CPU '
          'để tiết kiệm quota; Stage 5 trở đi bắt buộc phải có GPU.')


In [ ]:
#@title Tự dò thư mục dữ liệu
# Colab : dữ liệu hay nằm trong "Shared with me" -> phải dò qua shortcut-targets.
# Kaggle: dữ liệu nằm ở /kaggle/input/<tên-dataset>/... -> tên dataset do bạn đặt
#         nên cũng không hardcode được. Cùng một hàm dò xử lý cả hai.
import os, glob

def find_data_dir(roots, max_depth=6):
    """Thư mục đầu tiên chứa đủ DATA_FILES."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip('/').count('/')
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count('/') - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames
                           if not d.startswith('.') and d != '__MACOSX']
            if all(f in filenames for f in DATA_FILES):
                return dirpath
    return None

def data_roots():
    if ON_COLAB:
        return ['/content/drive/MyDrive',
                '/content/drive/.shortcut-targets-by-id',   # nơi shortcut trỏ tới
                '/content/drive/Shareddrives',
                '/content']
    if ON_KAGGLE:
        return ['/kaggle/input']                            # mọi dataset đã attach
    return ['.', '..']

if RAW is None or not all(os.path.exists(f'{RAW}/{f}') for f in DATA_FILES):
    print('đang dò dữ liệu...', flush=True)
    RAW = find_data_dir(data_roots())

assert RAW, (
    'KHÔNG tìm thấy dữ liệu (cần train.json + public-official.json).\n\n'
    'Colab: thư mục "UIT DS Challenge" thường ở "Shared with me" mà Colab không\n'
    '  mount phần đó. Sửa 1 lần: drive.google.com -> Shared with me -> chuột phải\n'
    '  "UIT DS Challenge" -> Organise -> Add shortcut to Drive -> My Drive.\n\n'
    'Kaggle: phải Add Input dataset chứa dữ liệu Task 1 vào notebook\n'
    '  (Notebook -> Add Input -> Datasets). Kiểm tra: !ls /kaggle/input\n\n'
    'Hoặc gán tay: RAW = "<đường dẫn tới thư mục LegalR - Public Test>"')

print('RAW =', RAW)
for f in DATA_FILES:
    p = f'{RAW}/{f}'
    print(f'  {f:26s} {os.path.getsize(p)/1e6:8.2f} MB')

# corpus: có thể là zip, hoặc đã bung sẵn (Kaggle dataset thường bung sẵn)
CTX_ZIP = f'{RAW}/selected-contexts.zip'
CTX_DIR = f'{SCRATCH}/contexts'
n_here  = len(glob.glob(f'{RAW}/**/context_*.json', recursive=True))
if n_here:
    CTX_DIR = RAW                     # đã bung sẵn cạnh dữ liệu -> đọc trực tiếp
    print(f'  context_*.json           {n_here} file (đã bung sẵn)')
elif os.path.exists(CTX_ZIP):
    print(f'  selected-contexts.zip    {os.path.getsize(CTX_ZIP)/1e6:8.2f} MB'
          ' -> Stage 1 sẽ giải nén vào', CTX_DIR)
else:
    print('  CẢNH BÁO: không thấy cả zip lẫn folder context_*.json')


---
## Stage 1 — Build corpus

Đọc 8.532 file `context_*.json` → **một** file parquet.

Hai vấn đề dữ liệu được xử lý ở đây:
- Zip giải nén **lồng hai lớp** (`selected-contexts/selected-contexts/...`) → dùng glob đệ quy.
- **1.125/8.532 văn bản (13%) thiếu field `name`** → lấy lại tiêu đề từ slug trong `link`.
  Tiêu đề là tín hiệu rất mạnh (chứa loại văn bản + chủ đề + năm), bỏ 13% là mất điểm thật.

Giải nén vào `/content` (đĩa local) chứ **không** vào Drive — ghi 8.532 file nhỏ lên Drive
mất hàng giờ.

In [ ]:
#@title Build corpus.parquet
import json, re, glob, os, zipfile, time
import pandas as pd

CORPUS_PQ = f'{ART}/corpus.parquet'

def _title_from_link(link):
    slug = (link or '').rstrip('/').rsplit('/', 1)[-1]
    slug = re.sub(r'\.aspx$', '', slug, flags=re.I)
    return slug.replace('-', ' ').strip()

def build_corpus():
    raw_dir  = globals().get('CTX_DIR', f'{SCRATCH}/contexts')
    zip_path = globals().get('CTX_ZIP', f'{RAW}/selected-contexts.zip')
    files = glob.glob(f'{raw_dir}/**/context_*.json', recursive=True)
    if not files:
        # thử folder đã giải nén sẵn cạnh dữ liệu gốc
        files = glob.glob(f'{RAW}/**/context_*.json', recursive=True)
    if not files:
        assert os.path.exists(zip_path), f'Không thấy {zip_path} — hãy upload lên Drive.'
        print('Giải nén zip vào đĩa tạm...', flush=True)
        os.makedirs(raw_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(raw_dir)
        files = glob.glob(f'{raw_dir}/**/context_*.json', recursive=True)
    print(f'{len(files)} file context', flush=True)

    rows, n_noname = [], 0
    for i, fp in enumerate(files):
        with open(fp, encoding='utf-8') as f:
            d = json.load(f)
        title = (d.get('name') or '').replace('-', ' ').strip()
        if not title:
            n_noname += 1
            title = _title_from_link(d.get('link', ''))
        # gộp mọi whitespace: passage có rất nhiều '\r\n\n  ' rác.
        # Làm sớm để regex tách 'Điều N' ở Stage 2 hoạt động ổn định.
        passage = re.sub(r'\s+', ' ', d.get('passage') or '').strip()
        rows.append({'doc_id': str(d['id']), 'title': title,
                     'link': d.get('link', ''), 'passage': passage})
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(files)}', flush=True)

    df = pd.DataFrame(rows).drop_duplicates('doc_id').reset_index(drop=True)
    print(f'-> {len(df)} doc | thiếu name (đã vá từ link): {n_noname}')
    df.to_parquet(CORPUS_PQ, index=False)
    return df

t0 = time.time()
_pq = cache_path('corpus.parquet')      # ART trước, rồi tới dataset chỉ đọc (Kaggle)
if _pq:
    corpus = pd.read_parquet(_pq)
    print(f'load cache: {len(corpus)} doc <- {_pq}')
else:
    corpus = build_corpus()

print('%.1fs' % (time.time() - t0))
print('passage chars: mean %d | p50 %d | max %d' % (
    corpus.passage.str.len().mean(), corpus.passage.str.len().median(),
    corpus.passage.str.len().max()))
corpus.head(3)[['doc_id', 'title']]

In [ ]:
#@title Load train / public + val split cố định
import json, random

with open(f'{RAW}/train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(f'{RAW}/public-official.json', encoding='utf-8') as f:
    public_raw = json.load(f)

# answer có thể là int hoặc str tuỳ file -> luôn ép str
QUESTIONS = {qid: v['question'] for qid, v in train_raw.items()}
GOLD = {qid: set(map(str, v['answer'])) for qid, v in train_raw.items()}
PUBLIC = {qid: v['question'] for qid, v in public_raw.items()}

# Split cố định theo seed + sorted key -> mọi session cho cùng một val set.
# Đây là điều kiện bắt buộc để so sánh các thí nghiệm với nhau.
all_qids = sorted(train_raw.keys())
random.Random(SEED).shuffle(all_qids)
VAL_QIDS   = all_qids[:N_VAL]
TRAIN_QIDS = all_qids[N_VAL:]

print(f'train {len(TRAIN_QIDS)} | val {len(VAL_QIDS)} | public {len(PUBLIC)}')

# kiểm tra gold có nằm trong corpus không
corpus_ids = set(corpus.doc_id)
missing = {g for q in all_qids for g in GOLD[q]} - corpus_ids
print('gold doc thiếu trong corpus:', len(missing))

---
## Stage 2 — Chunking theo cấu trúc pháp quy

Passage trung bình **41.000 ký tự** mà bi-encoder chỉ nhận ~256 token (~1.000 ký tự).
Không chunk là không dùng được dense retrieval.

Ba quyết định thiết kế ở đây:
- **Cắt theo `Điều N`** thay vì cắt cứng theo độ dài — ranh giới ngữ nghĩa của văn bản
  pháp luật nằm đúng ở đó. Điều ngắn thì gộp lại, Điều dài thì cắt tiếp có overlap.
- **Prepend tiêu đề vào mọi chunk.** Câu hỏi rất ngắn (trung bình 20 từ) nên tiêu đề
  văn bản là tín hiệu định chủ đề cực mạnh.
- **Chunk đầu tiên = tiêu đề + phần mở đầu**, nơi chứa số hiệu, cơ quan ban hành, trích yếu.
- **Cap `MAX_CHUNKS_PER_DOC`**: nếu không, vài văn bản 6M ký tự sẽ chiếm phần lớn index
  và bóp méo điểm `max` khi gộp chunk→doc.

In [ ]:
#@title Build chunks.parquet
import re, os, time
import pandas as pd

CHUNKS_REL = f'chunks_c{CHUNK_CHARS}.parquet'
CHUNKS_PQ  = f'{ART}/{CHUNKS_REL}'

DIEU_RE = re.compile(r'(?=Điều\s+\d+\s*[\.\:])')

def _split_long(s, size, overlap):
    """Cắt s thành các đoạn <= size ký tự, ưu tiên cắt ở ranh giới câu."""
    out, i = [], 0
    while i < len(s):
        j = min(i + size, len(s))
        if j < len(s):
            # lùi về dấu chấm gần nhất trong 200 ký tự cuối cho khỏi cắt giữa câu
            cut = s.rfind('. ', i + size - 200, j)
            if cut > i:
                j = cut + 1
        out.append(s[i:j].strip())
        if j >= len(s):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if c]

def chunk_doc(title, passage):
    """-> list các đoạn text (chưa gắn tiêu đề)."""
    pieces = [p.strip() for p in DIEU_RE.split(passage) if p.strip()]
    if not pieces:
        pieces = [passage]

    merged = []
    buf = ''
    for p in pieces:
        if len(p) > CHUNK_CHARS:
            if buf:
                merged.append(buf); buf = ''
            merged.extend(_split_long(p, CHUNK_CHARS, CHUNK_OVERLAP))
        elif len(buf) + len(p) + 1 <= CHUNK_CHARS:
            buf = f'{buf} {p}'.strip()
        else:
            merged.append(buf); buf = p
    if buf:
        merged.append(buf)

    # chunk mở đầu: chứa số hiệu / cơ quan ban hành / trích yếu
    head = passage[:CHUNK_CHARS].strip()
    if head and (not merged or merged[0][:200] != head[:200]):
        merged.insert(0, head)
    return merged[:MAX_CHUNKS_PER_DOC]

def build_chunks(corpus):
    rows = []
    for n, r in enumerate(corpus.itertuples(index=False)):
        for c in chunk_doc(r.title, r.passage):
            # tiêu đề đi kèm mọi chunk -> chunk lẻ vẫn biết mình thuộc văn bản nào
            rows.append((r.doc_id, f'{r.title}. {c}'))
        if (n + 1) % 1000 == 0:
            print(f'  {n+1}/{len(corpus)} doc -> {len(rows)} chunk', flush=True)
    df = pd.DataFrame(rows, columns=['doc_id', 'text'])
    df.to_parquet(CHUNKS_PQ, index=False)
    return df

t0 = time.time()
_cpq = cache_path(CHUNKS_REL)
if _cpq:
    chunks = pd.read_parquet(_cpq)
    print('load cache <-', _cpq)
else:
    chunks = build_chunks(corpus)
print('%d chunk | %.1fs' % (len(chunks), time.time() - t0))

# map chunk -> doc dạng số nguyên, dùng suốt phần sau
DOC_IDS = corpus.doc_id.tolist()
DOC2IX  = {d: i for i, d in enumerate(DOC_IDS)}
import numpy as np
CHUNK2DOC = np.array([DOC2IX[d] for d in chunks.doc_id], dtype=np.int32)
print('chunk/doc trung bình: %.1f' % (len(chunks) / len(corpus)))
chunks.text.str.len().describe()[['mean', '50%', 'max']]

---
## Stage 3 — Hạ tầng đánh giá

**Làm cái này trước khi tối ưu bất cứ gì.** `public-official.json` không có đáp án và chỉ
được nộp 10 lần/ngày, nên mọi quyết định kỹ thuật phải ra từ val set này — miễn phí, không
giới hạn số lần thử.

Công thức theo đúng BTC: `Recall = mean_q |gold ∩ pred| / |gold|` với `pred` cắt ở 5 doc.

Luôn in cả `@5`, `@20`, `@100` vì chúng chẩn đoán hai bệnh khác nhau:
- `@100` thấp → **retriever** yếu (văn bản đúng không vào nổi danh sách ứng viên)
- `@100` cao mà `@5` thấp → **ranking** yếu (tìm ra rồi nhưng không đẩy lên top)

In [ ]:
#@title Metrics + experiment log
import numpy as np, pandas as pd, os, datetime

def official_score(sub, truth):
    """Bản sao CHÍNH XÁC của scoring.py mà BTC dùng trên Codabench.

    sub: {qid: {'answer': [id]}} — đúng định dạng file nộp
    truth: {qid: [id]}

    Ba chi tiết trong code của BTC mà đọc mô tả không thấy:
    1. Recall KHÔNG cắt top-5. Nó chỉ *chặn* ở len<=5: quá 5 id -> câu đó 0 điểm.
       Nộp đúng 5 id là tối ưu cho Recall.
    2. Số lượng key phải khớp tuyệt đối, lệch là raise Exception -> TOÀN BỘ bài nộp
       lỗi, không phải chỉ mất điểm câu đó.
    3. Thiếu key -> len(None) -> TypeError -> cũng sập toàn bài.
    """
    y_pred = {k: v['answer'] for k, v in sub.items()}
    if len(y_pred) != len(truth):
        raise Exception('Samples in predict not match with reference')
    ok = lambda k: 0 < len(y_pred[k]) <= TOPK_SUBMIT
    recall = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(truth[k])
                      if ok(k) else 0 for k in truth])
    precision = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(y_pred[k])
                         if ok(k) else 0 for k in y_pred])
    return {'recall': float(recall), 'precision': float(precision)}

def evaluate(preds, gold, ks=(5, 20, 100)):
    """preds: {qid: [doc_id đã xếp hạng]} -> recall@k chẩn đoán + điểm chính thức.

    recall@20/@100 KHÔNG phải metric của cuộc thi — chúng để chẩn đoán:
      @100 thấp        -> retriever yếu (văn bản đúng không vào nổi danh sách)
      @100 cao, @5 thấp -> ranking yếu (tìm ra rồi mà không đẩy lên top)
    """
    out = {}
    for k in ks:
        out[f'recall@{k}'] = float(np.mean([
            len(gold[q] & set(preds.get(q, [])[:k])) / len(gold[q]) for q in gold]))
    # điểm chính thức: dựng đúng shape file nộp rồi chấm bằng code của BTC
    sub = {q: {'answer': list(preds.get(q, [])[:TOPK_SUBMIT])} for q in gold}
    off = official_score(sub, {q: list(gold[q]) for q in gold})
    out['OFFICIAL_recall'] = off['recall']
    out['OFFICIAL_precision'] = off['precision']
    return out

def show(name, preds, gold=None, qids=None):
    gold = gold or {q: GOLD[q] for q in (qids or VAL_QIDS)}
    m = evaluate(preds, gold)
    print(name.ljust(28), ' '.join(f'{k}={v:.4f}' for k, v in m.items()))
    return m

LOG_CSV = f'{ART}/experiments.csv'

def log_exp(name, metrics, note=''):
    """Ghi lại mọi thí nghiệm. Bảng này là thứ bạn cần để chọn cấu hình cho private
    test và để viết paper nếu vào top 10 — đừng bỏ qua."""
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'),
           'profile': PROFILE, 'run': name, 'note': note, **metrics}
    df = pd.DataFrame([row])
    if os.path.exists(LOG_CSV):
        df = pd.concat([pd.read_csv(LOG_CSV), df], ignore_index=True)
    df.to_csv(LOG_CSV, index=False)
    return df.tail(5)

print('ok')


# ---- Đối chiếu official_score() với scoring.py THẬT của BTC ------------------
# Dataset uit-task1-scoring có sẵn mã chấm điểm Codabench -> nạp trực tiếp
# eval_retrieval() của BTC và so với bản chép tay ở trên trên vài trường hợp
# biên. Nếu lệch thì mọi con số OFFICIAL_* trong notebook đều vô nghĩa.
import importlib.util, glob as _glob

def _load_btc_eval():
    cands = ([f'{SCORING_DIR}/scoring.py'] if SCORING_DIR else []) + \
            sorted(_glob.glob('/kaggle/input/**/Scoring-Program-Task-LegalIR/scoring.py',
                              recursive=True))
    for p in cands:
        if not os.path.exists(p):
            continue
        spec = importlib.util.spec_from_file_location('btc_scoring', p)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)          # main() có guard __main__ -> import an toàn
        return mod.eval_retrieval, p
    return None, None

_btc_eval, _btc_path = _load_btc_eval()
if _btc_eval is None:
    print('không thấy scoring.py của BTC -> bỏ qua đối chiếu '
          '(Kaggle: Add Input dataset uit-task1-scoring)')
else:
    _truth = {'a': ['1'], 'b': ['2', '3'], 'c': ['4']}
    _cases = {
        'đúng 1/1':        {'a': ['1'], 'b': ['2', '3'], 'c': ['4']},
        'nộp 5 id':        {'a': ['1', 'x', 'y', 'z', 'w'], 'b': ['2', '3', 'x', 'y', 'z'],
                            'c': ['q', 'w', 'e', 'r', 't']},
        'nộp 6 id -> 0':   {'a': ['1', 'x', 'y', 'z', 'w', 'v'], 'b': ['2'], 'c': ['4']},
    }
    _ok = True
    for _name, _p in _cases.items():
        _mine = official_score({k: {'answer': v} for k, v in _p.items()}, _truth)
        _theirs = _btc_eval({k: {'answer': v} for k, v in _p.items()}, _truth)
        _same = (abs(_mine['recall'] - _theirs['recall']) < 1e-9 and
                 abs(_mine['precision'] - _theirs['precision']) < 1e-9)
        _ok &= _same
        print(f'  {_name:16s} recall={_theirs["recall"]:.4f} '
              f'precision={_theirs["precision"]:.4f}  {"khớp" if _same else "LỆCH!"}')
    assert _ok, 'official_score() lệch so với scoring.py của BTC — sửa trước khi tin số liệu'
    print('official_score() khớp scoring.py của BTC <-', _btc_path)


In [ ]:
#@title Gộp chunk -> doc bằng MAX, rồi lấy top-N doc
def chunks_to_docs(chunk_scores, chunk_top_ix, topn=CAND_DOCS):
    """Điểm doc = MAX điểm các chunk của nó (không phải mean).

    Gold là "văn bản CHỨA thông tin cần thiết" -> một chunk khớp là đủ. Lấy mean sẽ
    phạt oan các văn bản dài mà chỉ một Điều liên quan.
    """
    best = {}
    for ix, s in zip(chunk_top_ix, chunk_scores):
        d = CHUNK2DOC[ix]
        if s > best.get(d, -1e30):
            best[d] = s
    ranked = sorted(best.items(), key=lambda x: -x[1])[:topn]
    return [DOC_IDS[d] for d, _ in ranked], [s for _, s in ranked]

print('ok')

---
## Stage 4 — BM25 (chunk-level)

Không cần GPU — **chạy stage này trong session CPU** để tiết kiệm quota GPU, đó là tài
nguyên khan hiếm nhất của bạn.

Cài đặt BM25 dạng sparse matrix (`W @ q`) thay vì `rank_bm25`: với ~500k chunk thì
`rank_bm25` chậm tới mức không dùng được, còn sparse matmul chấm 1 query mất vài chục ms.

BM25 bù đúng chỗ dense yếu: thuật ngữ pháp lý hiếm (*"Kiểm ngư viên trung cấp"*,
*"hành lang an toàn kỹ thuật anten"*) mà embedding thường trượt hẳn.

In [ ]:
#@title BM25 sparse
import numpy as np, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

class SparseBM25:
    """BM25 Okapi tiền tính thành ma trận trọng số -> truy vấn = 1 phép matmul sparse."""

    def __init__(self, k1=BM25_K1, b=BM25_B, min_df=3, max_features=400_000):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(lowercase=True, min_df=min_df,
                                   max_features=max_features, dtype=np.int32)

    def fit(self, texts):
        X = self.vec.fit_transform(texts).tocsc()
        n, _ = X.shape
        dl = np.asarray(X.sum(axis=1)).ravel()
        avgdl = dl.mean()
        df = np.diff(X.indptr)
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

        X = X.tocoo()
        tf = X.data.astype(np.float32)
        denom = tf + self.k1 * (1 - self.b + self.b * dl[X.row] / avgdl)
        data = idf[X.col] * tf * (self.k1 + 1) / denom
        self.W = sp.csr_matrix((data, (X.row, X.col)), shape=(n, len(idf),))
        self.W = self.W.T.tocsr()      # (vocab, n_chunk) -> query nhanh hơn
        return self

    def search(self, queries, topk=CAND_CHUNKS):
        Q = self.vec.transform(queries)
        Q.data[:] = 1.0               # query dùng binary term presence
        S = (Q.astype(np.float32) @ self.W).toarray()   # (nq, n_chunk)
        k = min(topk, S.shape[1])
        ix = np.argpartition(-S, k - 1, axis=1)[:, :k]
        srt = np.take_along_axis(S, ix, 1).argsort(axis=1)[:, ::-1]
        ix = np.take_along_axis(ix, srt, 1)
        return ix, np.take_along_axis(S, ix, 1)

import time, pickle
t0 = time.time()
bm25 = SparseBM25().fit(chunks.text.tolist())
print('BM25 fit: %.1fs | vocab %d' % (time.time() - t0, bm25.W.shape[0]))

In [ ]:
#@title Đánh giá BM25 trên val
def retrieve_bm25(qids, qtext, batch=64):
    preds = {}
    for i in range(0, len(qids), batch):
        bq, bt = qids[i:i+batch], qtext[i:i+batch]
        ix, sc = bm25.search(bt)
        for q, row_ix, row_sc in zip(bq, ix, sc):
            preds[q] = chunks_to_docs(row_sc, row_ix)[0]
    return preds

t0 = time.time()
bm25_val = retrieve_bm25(VAL_QIDS, [QUESTIONS[q] for q in VAL_QIDS])
print('%.1fs' % (time.time() - t0))
m = show('BM25 chunk-level', bm25_val)
log_exp('bm25', m, f'chunk={CHUNK_CHARS} k1={BM25_K1} b={BM25_B}')

---
## Stage 5 — Dense retrieval

Bước nặng duy nhất. **Corpus là cố định và dùng chung cho cả train lẫn public test** →
encode **một lần**, tái dùng cho mọi thí nghiệm sau. Đây là điểm tiết kiệm lớn nhất trên Colab.

Encode theo **shard 50k chunk, ghi từng shard ra Drive ngay**. Session Colab chết ở phút
thứ 50 mà không có shard là mất trắng cả tiếng GPU; có shard thì chỉ chạy lại phần thiếu.

Vector lưu **fp16** (~500k × 768 × 2B ≈ 750MB) — vừa RAM, vừa VRAM T4, không cần faiss:
`q @ emb.T` trên GPU đã đủ nhanh và khỏi phải build index.

In [ ]:
#@title Encode corpus (đa GPU, sharded, resume được)
import numpy as np, os, glob, time, re, torch
from sentence_transformers import SentenceTransformer

def set_dense_model(model_path):
    """Chọn checkpoint cho nhánh dense + trỏ EMB_DIR sang cache của đúng checkpoint đó.

    Tách thành hàm vì Stage 8 gọi lại nó sau khi fine-tune: model đổi thì embedding
    cũ vô giá trị, phải đổi cả nơi ghi lẫn nơi đọc chứ không chỉ đổi mỗi model.
    """
    global MODEL_PATH, TAG, EMB_REL, EMB_DIR, EMB_DIRS
    MODEL_PATH = model_path
    TAG     = re.sub(r'[^a-zA-Z0-9]+', '_', MODEL_PATH) + f'_c{CHUNK_CHARS}'
    EMB_REL = f'emb/{TAG}'
    EMB_DIR = f'{ART}/{EMB_REL}'          # nơi GHI
    os.makedirs(EMB_DIR, exist_ok=True)
    # nơi ĐỌC: thư mục ghi + mọi cache chỉ đọc (Kaggle: output session trước attach lại)
    EMB_DIRS = [EMB_DIR] + [f'{d}/{EMB_REL}' for d in ART_IN if os.path.isdir(f'{d}/{EMB_REL}')]
    print('model    ->', MODEL_PATH)
    print('ghi      ->', EMB_DIR)
    if len(EMB_DIRS) > 1:
        print('đọc thêm <-', EMB_DIRS[1:])

def _shard_file(a):
    """shard đã encode ở BẤT KỲ cache nào -> path, chưa có -> None."""
    for d in EMB_DIRS:
        p = f'{d}/s_{a:08d}.npy'
        if os.path.exists(p):
            return p
    return None

def _l2(v):
    """normalize tay: encode_multi_process ở một số bản ST không nhận
    normalize_embeddings, làm ở đây thì mọi bản đều cho ra vector cùng thang."""
    v = np.asarray(v, dtype=np.float32)
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v

def encode_corpus():
    texts  = chunks.text.tolist()
    shards = [(i, min(i + ENC_SHARD, len(texts))) for i in range(0, len(texts), ENC_SHARD)]
    todo   = [(a, b) for a, b in shards if _shard_file(a) is None]

    if todo:
        model = SentenceTransformer(MODEL_PATH, device=DEV)
        model.max_seq_length = MAX_SEQ
        if torch.cuda.is_available():
            model.half()

        # >1 GPU: mỗi GPU một process con, ST tự chia đều sentence cho các process.
        # Đây là chỗ ăn tốc độ lớn nhất của Kaggle 2×GPU — encode là bước nặng nhất.
        pool = None
        if len(DEVICES) > 1:
            try:
                pool = model.start_multi_process_pool(target_devices=DEVICES)
                print(f'multi-GPU pool: {DEVICES}', flush=True)
            except Exception as e:
                print('không mở được multi-process pool (%s) -> chạy 1 GPU' % e, flush=True)
                pool = None

        print(f'{len(todo)}/{len(shards)} shard cần encode', flush=True)
        try:
            for a, b in todo:
                t0 = time.time()
                if pool is not None:
                    v = model.encode_multi_process(
                        texts[a:b], pool, batch_size=ENC_BATCH,
                        chunk_size=max(500, (b - a) // (4 * len(DEVICES))))
                else:
                    v = model.encode(texts[a:b], batch_size=ENC_BATCH,
                                     convert_to_numpy=True, show_progress_bar=False)
                np.save(f'{EMB_DIR}/s_{a:08d}.npy', _l2(v).astype(np.float16))
                print(f'  shard {a}-{b}: {time.time()-t0:.0f}s', flush=True)
        finally:
            if pool is not None:
                model.stop_multi_process_pool(pool)
            del model
            torch.cuda.empty_cache()

    return np.concatenate([np.load(_shard_file(a)) for a, _ in shards])

set_dense_model(FT_BIENCODER_PATH if USE_FINETUNED else BIENCODER)
t0  = time.time()
EMB = encode_corpus()
assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)} — xoá {EMB_DIR} và encode lại'
print('EMB', EMB.shape, EMB.dtype, '| %.1f MB | %.1fs' % (EMB.nbytes / 1e6, time.time() - t0))


In [ ]:
#@title Dense search trên GPU
import torch, numpy as np
from sentence_transformers import SentenceTransformer

# EMB fp16 chỉ vài trăm MB nên để trọn trên 1 GPU: q @ EMB.T là một phép matmul,
# chia cho 2 GPU chỉ thêm đồng bộ chứ không nhanh hơn đáng kể. GPU còn lại được
# dùng thật sự ở Stage 5 (encode) và Stage 7 (rerank) — hai bước nặng.
EMB_T  = None
qmodel = None

def free_dense():
    """Nhả EMB_T + query encoder khỏi VRAM. Stage 8 gọi trước khi train: chỗ này
    đang giữ vài GB, để nguyên thì fine-tune gần như chắc chắn OOM."""
    global EMB_T, qmodel
    EMB_T = qmodel = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_query_model():
    """Đưa EMB lên GPU + nạp encoder cho query theo MODEL_PATH hiện tại.
    Gọi lại được — Stage 8 dùng để chuyển sang model vừa fine-tune."""
    global EMB_T, qmodel
    free_dense()
    EMB_T = torch.from_numpy(EMB).to(DEV)                   # (N, dim) fp16
    qmodel = SentenceTransformer(MODEL_PATH, device=DEV)
    qmodel.max_seq_length = MAX_SEQ
    if DEV.startswith('cuda'):
        qmodel.half()

def dense_search(qtext, topk=CAND_CHUNKS, batch=32):
    """-> (indices, scores) của top chunk cho mỗi query."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        q = qmodel.encode(qtext[i:i+batch], batch_size=batch, normalize_embeddings=True,
                          convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        s = q @ EMB_T.T                                     # cosine (đã normalize)
        v, ix = torch.topk(s, min(topk, s.shape[1]), dim=1)
        ixs.append(ix.cpu().numpy()); scs.append(v.float().cpu().numpy())
    return np.concatenate(ixs), np.concatenate(scs)

def retrieve_dense(qids, qtext):
    ix, sc = dense_search(qtext)
    return {q: chunks_to_docs(s, i)[0] for q, i, s in zip(qids, ix, sc)}

load_query_model()
t0 = time.time()
val_text = [QUESTIONS[q] for q in VAL_QIDS]
dense_val = retrieve_dense(VAL_QIDS, val_text)
print('%.1fs' % (time.time() - t0))
m = show('dense' + ('-ft' if USE_FINETUNED else ''), dense_val)
log_exp('dense_ft' if USE_FINETUNED else 'dense', m, MODEL_PATH)


---
## Stage 6 — Hybrid fusion (RRF)

Dùng **Reciprocal Rank Fusion** thay vì cộng điểm có trọng số: điểm BM25 và điểm cosine
nằm trên hai thang hoàn toàn khác nhau, chuẩn hoá chúng luôn mong manh. RRF chỉ dùng
**hạng**, không cần tune trọng số, và bền hơn nhiều.

`score(d) = Σ 1 / (K + rank_i(d))`

In [ ]:
#@title RRF fuse
def rrf(*rankings, k=RRF_K, topn=CAND_DOCS):
    """Mỗi ranking: {qid: [doc_id]} -> {qid: [doc_id] đã fuse}."""
    out = {}
    for q in rankings[0]:
        s = {}
        for r in rankings:
            for rank, d in enumerate(r.get(q, [])):
                s[d] = s.get(d, 0.0) + 1.0 / (k + rank + 1)
        out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])[:topn]]
    return out

hybrid_val = rrf(bm25_val, dense_val)
m = show('hybrid RRF', hybrid_val)
log_exp('hybrid_rrf', m, f'K={RRF_K}')

---
## Stage 7 — Cross-encoder rerank

**Đây là bước ăn điểm mạnh nhất.** Baseline cho `recall@100 = 0.87` nhưng `recall@5 = 0.486`
— khoảng cách đó chính là dư địa của reranking, và cross-encoder tồn tại để thu hẹp nó.

Cross-encoder đọc `(question, chunk)` cùng lúc nên hiểu quan hệ sâu hơn bi-encoder, nhưng
đắt gấp hàng nghìn lần → chỉ chấm `RERANK_TOP=50` doc, mỗi doc đại diện bởi **chunk tốt nhất**
theo dense score.

Corpus chỉ 8.532 văn bản nên toàn bộ 1.000 câu public test × 50 ứng viên = 50k cặp,
khoảng 5–10 phút trên T4. Hoàn toàn khả thi.

### Rerank không mặc nhiên có lợi — phải đo

Lần chạy thử đầu tiên trên Kaggle, corpus rút gọn 2.285 doc / 200 câu val:

| | recall@5 | recall@20 |
|---|---|---|
| hybrid RRF | **0.9192** | 0.9700 |
| + rerank (replace) | 0.8583 | 0.9725 |

Reranker kéo `@20` lên nhưng làm hỏng `@5` — đúng chỗ tính điểm. Hai nguyên nhân,
đã xử lý cả hai:

1. **Chunk đại diện chọn sai.** Cross-encoder chỉ đọc một chunk cho mỗi doc. Doc lọt
   vào ứng viên nhờ BM25 thường không có chunk nào trong top dense, trước đây bị gán
   đại chunk đầu văn bản → reranker chấm một đoạn không liên quan rồi đạp doc xuống.
   Stage 7 giờ lấy chunk BM25 tốt nhất cho đúng nhóm doc đó.
2. **Tin reranker 100%.** `RERANK_MODE` ở Stage 0 chọn giữa `rrf` (mặc định, RRF giữa
   thứ tự hybrid và thứ tự rerank), `replace` (cách cũ) và `none` (bỏ hẳn rerank).

Cell rerank in điểm val của **cả ba** trong một lần chạy GPU rồi nói thẳng cái nào
thắng. Con số ở trên đo trên corpus rút gọn — đọc lại bảng đó ở lần chạy đầy đủ
(8.532 doc, 1.000 câu val) rồi mới chốt, vì bài toán khó lên thì cán cân có thể đổi.


In [ ]:
#@title Chọn chunk đại diện cho mỗi doc ứng viên
import numpy as np
from collections import defaultdict

def bm25_topk(qtext, batch=64):
    """bm25.search() dựng ma trận dense (nq × n_chunk) -> phải chấm theo lô,
    1000 query × ~500k chunk một lần là ~2GB RAM."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        ix, sc = bm25.search(qtext[i:i+batch])
        ixs.append(ix); scs.append(sc)
    return np.concatenate(ixs), np.concatenate(scs)

def best_chunk_per_doc(qids, qtext, cand):
    """{qid: {doc_id: chunk tốt nhất}} — hỏi CẢ hai retriever.

    Cross-encoder chỉ được đọc MỘT chunk cho mỗi doc, nên chunk đó chọn sai là
    doc chết oan. Doc lọt vào danh sách ứng viên nhờ BM25 thường không có chunk
    nào trong top dense; lấy đại chunk đầu văn bản (toàn phần tiêu đề / căn cứ
    pháp lý) là đưa cho reranker một đoạn không liên quan -> nó đạp doc đó
    xuống, kể cả khi đó là gold. Vì vậy:
        1. có chunk trong top dense  -> lấy chunk dense điểm cao nhất
        2. không có                  -> lấy chunk BM25 điểm cao nhất
        3. vẫn không có (hiếm)       -> đành lấy chunk đầu
    """
    d_ix, d_sc = dense_search(qtext, topk=CAND_CHUNKS)
    b_ix, b_sc = bm25_topk(qtext)

    best = {}
    for n, q in enumerate(qids):
        want, top = set(cand[q]), {}
        # 1. dense
        for ci, s in zip(d_ix[n], d_sc[n]):
            d = DOC_IDS[CHUNK2DOC[ci]]
            if d in want and s > top.get(d, (-1e30, 0))[0]:
                top[d] = (s, ci)
        # 2. BM25 cho phần doc còn thiếu (thang điểm khác dense -> chỉ dùng để
        #    so BM25 với BM25, không trộn chung)
        miss = want - set(top)
        if miss:
            bt = {}
            for ci, s in zip(b_ix[n], b_sc[n]):
                d = DOC_IDS[CHUNK2DOC[ci]]
                if d in miss and s > bt.get(d, (-1e30, 0))[0]:
                    bt[d] = (s, ci)
            top.update(bt)
        # 3. hết cách
        for d in want - set(top):
            first = int(np.searchsorted(CHUNK2DOC, DOC2IX[d]))
            top[d] = (-1e30, first)
        best[q] = {d: ci for d, (_, ci) in top.items()}
    return best

CHUNK_TEXTS = chunks.text.tolist()
print('ok')


In [ ]:
#@title Rerank (chia query cho từng GPU)
from sentence_transformers import CrossEncoder
import numpy as np, time, torch, threading

RR_DEVICES = DEVICES                       # 1 reranker / GPU
RERANKERS  = {}

def get_reranker(dev):
    if dev not in RERANKERS:
        m = CrossEncoder(RERANKER, max_length=RERANK_MAXLEN, device=dev)
        if RERANK_FP16 and dev.startswith('cuda'):
            m.model.half()
        RERANKERS[dev] = m
    return RERANKERS[dev]

# Nạp trước, tuần tự: hai thread cùng tải model một lúc sẽ tranh cache HuggingFace.
for _d in RR_DEVICES:
    get_reranker(_d)
    print('reranker ->', _d, flush=True)

def rerank(qids, qtext, cand, top=RERANK_TOP):
    """Mỗi GPU nhận một phần danh sách query và chạy độc lập trong một thread.
    Forward của HF nhả GIL khi chờ CUDA nên 2 thread thực sự chạy song song."""
    rep = best_chunk_per_doc(qids, qtext, {q: cand[q][:top] for q in qids})

    jobs = list(zip(qids, qtext))
    parts = [jobs[i::len(RR_DEVICES)] for i in range(len(RR_DEVICES))]
    out, lock, done = {}, threading.Lock(), [0]

    def work(dev, part):
        m = get_reranker(dev)
        for q, qt in part:
            docs = cand[q][:top]
            if docs:
                s = m.predict([[qt, CHUNK_TEXTS[rep[q][d]]] for d in docs],
                              batch_size=RERANK_BATCH, show_progress_bar=False)
                order = np.argsort(-np.asarray(s))
                # doc ngoài top rerank vẫn giữ nguyên thứ tự cũ ở phía sau
                ranked = [docs[i] for i in order] + cand[q][top:]
            else:
                ranked = []
            with lock:
                out[q] = ranked
                done[0] += 1
                if done[0] % 100 == 0:
                    print(f'  {done[0]}/{len(qids)}', flush=True)

    ths = [threading.Thread(target=work, args=(d, p), daemon=True)
           for d, p in zip(RR_DEVICES, parts) if p]
    for t in ths: t.start()
    for t in ths: t.join()
    return {q: out[q] for q in qids}

def fuse_rerank(cand, reranked, k=RRF_K):
    """RRF giữa thứ tự hybrid và thứ tự sau rerank.

    'replace' đặt toàn bộ niềm tin vào cross-encoder: nó xếp nhầm một câu là câu
    đó mất trắng, dù retriever đã để gold ở hạng 1. RRF giữ lại phiếu của
    retriever nên hỏng ít hơn — đổi lại phần thắng cũng nhỏ hơn khi reranker đúng.
    """
    out = {}
    for q, docs in reranked.items():
        s = {}
        for r, d in enumerate(cand[q]):
            s[d] = s.get(d, 0.0) + 1.0 / (k + r + 1)
        for r, d in enumerate(docs):
            s[d] = s.get(d, 0.0) + 1.0 / (k + r + 1)
        out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])]
    return out

def apply_rerank(qids, qtext, cand, mode=None):
    """Một cửa duy nhất cho cả val lẫn public -> hai bên không thể lệch nhau."""
    mode = mode or RERANK_MODE
    if mode == 'none':
        return cand
    r = rerank(qids, qtext, cand)
    return r if mode == 'replace' else fuse_rerank(cand, r)

t0 = time.time()
rerank_val = rerank(VAL_QIDS, val_text, hybrid_val)
print('%.1fs (%d GPU)' % (time.time() - t0, len(RR_DEVICES)))

# In cả ba lựa chọn trên cùng một lần chạy reranker (fuse không tốn thêm GPU).
# Chọn RERANK_MODE ở Stage 0 theo bảng này, đừng chọn theo cảm giác.
m_none    = show('[none]    hybrid RRF', hybrid_val)
m_replace = show('[replace] rerank thay hẳn', rerank_val)
rerank_rrf_val = fuse_rerank(hybrid_val, rerank_val)
m_rrf     = show('[rrf]     rerank fuse RRF', rerank_rrf_val)

_note = f'{RERANKER} top{RERANK_TOP} fp16={RERANK_FP16} ngpu={len(RR_DEVICES)}'
log_exp('rerank_none', m_none, _note)
log_exp('rerank_replace', m_replace, _note)
log_exp('rerank_rrf', m_rrf, _note)

_best = max([('none', m_none), ('replace', m_replace), ('rrf', m_rrf)],
            key=lambda x: (x[1]['OFFICIAL_recall'], x[1]['OFFICIAL_precision']))[0]
print(f"\n-> val chọn RERANK_MODE = '{_best}' (đang đặt: '{RERANK_MODE}')")
if _best != RERANK_MODE:
    print('   sửa RERANK_MODE ở Stage 0 rồi chạy lại cell Stage 9 (không cần encode lại).')


---
## Stage 8 — Fine-tune bi-encoder *(ROI cao nhất sau reranker)*

6.000 cặp (câu hỏi → văn bản gold) trong train split là tài sản lớn nhất bạn có.
Embedding off-the-shelf hiểu tiếng Việt phổ thông nhưng **không** hiểu văn phong pháp quy.

Quy trình **iterative hard negative mining**:
1. Chạy retrieval hiện tại trên train split → top-50 doc.
2. Bỏ gold ra → phần còn lại là **hard negative** (giống gold nhưng sai — đúng thứ model
   cần học phân biệt; random negative gần như vô dụng).
3. Train `MultipleNegativesRankingLoss`.
4. Chạy lại cell này → vòng 2 mine negative bằng chính model vừa train, thường thêm
   được vài điểm nữa.

Cell này chạy **trọn gói**, không phải sửa cờ thủ công nữa:

```
mine negative (model hiện tại) → nhả VRAM → train → encode lại corpus bằng model mới
→ nạp lại query encoder → đo lại val (dense / hybrid / rerank)
```

Sau khi nó chạy xong, `MODEL_PATH` đã trỏ vào checkpoint fine-tune nên **Stage 9
(public test) tự dùng model mới** — cứ chạy tiếp các cell phía dưới.

Bật/tắt bằng `DO_FINETUNE` ở **Stage 0**.

> **VRAM.** Cell tự gọi `free_dense()` + xoá cache cross-encoder trước khi train, nhưng
> batch vẫn là thứ quyết định: batch càng lớn MNRL càng mạnh (in-batch negatives).
> Nếu OOM thì giảm `FT_BATCH`, đừng giảm `max_seq_length`.
>
> **Encode lại tốn thời gian bằng đúng Stage 5** (embedding của model cũ không dùng lại
> được). Vì vậy hãy train xong hẳn rồi mới encode lại — đừng encode lại sau mỗi epoch.


In [ ]:
#@title Mine hard negatives + train + encode lại corpus
FT_BATCH   = 64
FT_MAXSEQ  = 256   # hạ so với MAX_SEQ để đổi lấy batch lớn hơn
FT_EPOCHS  = 2
DO_FINETUNE = globals().get('DO_FINETUNE', True)   # đặt ở Stage 0

# Stage này KHÁC hai stage trên: nhiều GPU không giúp, có khi còn hại.
# sentence-transformers .fit() đi qua HF Trainer, thấy 2 GPU là tự bọc
# nn.DataParallel -> batch bị cắt làm hai, MultipleNegativesRankingLoss tính
# in-batch negatives riêng trên từng nửa -> mỗi mẫu chỉ còn một nửa số negative.
# Cách xử lý: đặt VISIBLE_GPUS='0' ở Stage 0 + Restart runtime (khuyến nghị),
# hoặc chạy 2 GPU nhưng nhân đôi FT_BATCH để bù lại.
if DO_FINETUNE and N_GPU > 1:
    if FT_BATCH_X2_ON_MULTIGPU:
        FT_BATCH *= 2
        print(f'{N_GPU} GPU + DataParallel -> nhân đôi FT_BATCH = {FT_BATCH} để giữ '
              'số in-batch negative mỗi mẫu (có thể OOM: hạ FT_MAXSEQ nếu cần)')
    else:
        print(f'CẢNH BÁO: đang thấy {N_GPU} GPU, MNRL sẽ yếu đi vì DataParallel. '
              "Nên đặt VISIBLE_GPUS='0' ở Stage 0 rồi Restart runtime.")

if DO_FINETUNE:
    import random, shutil, gc, os, glob, time, torch, numpy as np
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader

    # ---- 1. mine hard negative bằng retrieval HIỆN TẠI -----------------------
    # Chạy lần 2 (sau khi đã có checkpoint) thì retrieval ở đây chính là model
    # fine-tune vòng 1 -> negative khó hơn -> đúng nghĩa "iterative mining".
    tr_qids = TRAIN_QIDS
    tr_text = [QUESTIONS[q] for q in tr_qids]
    print('mine hard negatives...', flush=True)
    cand = rrf(retrieve_bm25(tr_qids, tr_text), retrieve_dense(tr_qids, tr_text))
    rep  = best_chunk_per_doc(tr_qids, tr_text, cand)

    # chunk đại diện của chính gold doc, dùng làm positive
    gold_rep = best_chunk_per_doc(tr_qids, tr_text, {q: list(GOLD[q]) for q in tr_qids})

    rng = random.Random(SEED)
    examples = []
    for q in tr_qids:
        gold = GOLD[q]
        pos_ci = [gold_rep[q][g] for g in gold if g in gold_rep[q]]
        if not pos_ci:
            continue
        negs = [d for d in cand[q][:50] if d not in gold]
        if not negs:
            continue
        for nd in rng.sample(negs, min(4, len(negs))):
            examples.append(InputExample(texts=[QUESTIONS[q],
                                                CHUNK_TEXTS[rng.choice(pos_ci)],
                                                CHUNK_TEXTS[rep[q][nd]]]))
    print(f'{len(examples)} training triplet')
    assert examples, 'không mine được triplet nào — kiểm tra GOLD / TRAIN_QIDS'

    # ---- 2. nhả VRAM trước khi train ----------------------------------------
    # EMB_T (corpus embedding), query encoder và các cross-encoder đang giữ vài GB.
    # Không nhả thì train batch lớn là OOM ngay. EMB (numpy) cũng bỏ luôn vì lát
    # nữa encode lại từ đầu bằng model mới.
    free_dense()
    RERANKERS.clear()
    EMB = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- 3. train ------------------------------------------------------------
    # Luôn train từ checkpoint GỐC, không train chồng lên bản fine-tune trước:
    # train chồng nhiều vòng trên cùng 6k câu hỏi là đường thẳng tới overfit.
    t0 = time.time()
    model = SentenceTransformer(BIENCODER, device=DEV)
    model.max_seq_length = FT_MAXSEQ
    dl = DataLoader(examples, shuffle=True, batch_size=FT_BATCH, drop_last=True)
    model.fit(train_objectives=[(dl, losses.MultipleNegativesRankingLoss(model))],
              epochs=FT_EPOCHS, warmup_steps=int(0.1 * len(dl) * FT_EPOCHS),
              use_amp=True, output_path=FT_BIENCODER_PATH, show_progress_bar=True)
    print('train %.1f phút | đã lưu -> %s' % ((time.time() - t0) / 60, FT_BIENCODER_PATH))
    del model, dl, examples
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- 4. encode lại corpus bằng model mới --------------------------------
    USE_FINETUNED = True
    set_dense_model(FT_BIENCODER_PATH)
    if glob.glob(f'{EMB_DIR}/s_*.npy'):
        # Checkpoint fine-tune luôn ghi vào cùng một path -> TAG không đổi -> shard
        # còn lại là của LẦN TRAIN TRƯỚC. Dùng lại là đang search bằng embedding của
        # model cũ trong khi query đi qua model mới: sai lệch mà không báo lỗi.
        print('xoá embedding của lần fine-tune trước:', EMB_DIR, flush=True)
        shutil.rmtree(EMB_DIR, ignore_errors=True)
        os.makedirs(EMB_DIR, exist_ok=True)
    EMB_DIRS = [EMB_DIR]      # cache chỉ-đọc cũng là của model cũ -> không đọc
    t0  = time.time()
    EMB = encode_corpus()
    assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)}'
    print('EMB', EMB.shape, '| %.1f MB | %.1f phút' % (EMB.nbytes / 1e6, (time.time() - t0) / 60))
    load_query_model()

    # ---- 5. đo lại trên val --------------------------------------------------
    # Ghi đè dense_val / hybrid_val / rerank_val để Stage 9 và bảng thí nghiệm
    # nói về cùng một model.
    dense_val  = retrieve_dense(VAL_QIDS, val_text)
    m = show('[ft] dense', dense_val)
    log_exp('dense_ft', m, FT_BIENCODER_PATH)

    hybrid_val = rrf(bm25_val, dense_val)
    m_none = show('[ft] hybrid RRF', hybrid_val)
    log_exp('hybrid_rrf_ft', m_none, f'K={RRF_K}')

    rerank_val     = rerank(VAL_QIDS, val_text, hybrid_val)
    m_replace      = show('[ft] rerank replace', rerank_val)
    rerank_rrf_val = fuse_rerank(hybrid_val, rerank_val)
    m_rrf          = show('[ft] rerank fuse RRF', rerank_rrf_val)
    _note = f'ft {RERANKER} top{RERANK_TOP}'
    log_exp('rerank_replace_ft', m_replace, _note)
    log_exp('rerank_rrf_ft', m_rrf, _note)

    _best = max([('none', m_none), ('replace', m_replace), ('rrf', m_rrf)],
                key=lambda x: (x[1]['OFFICIAL_recall'], x[1]['OFFICIAL_precision']))[0]
    print(f"\n-> val (sau fine-tune) chọn RERANK_MODE = '{_best}' (đang đặt: '{RERANK_MODE}')")
    print('MODEL_PATH giờ là bản fine-tune -> Stage 9 chạy tiếp là tự dùng model mới.')
else:
    print('DO_FINETUNE=False — bỏ qua (Stage 9 sẽ dùng', MODEL_PATH, ').')


---
## Stage 9 — Dự đoán public test + validate + đóng gói

Validator chặn đúng những lỗi làm mất điểm oan:
- Thiếu qid so với `public-official.json`
- Câu có **> 5** id → BTC cho câu đó **0 điểm cả Recall lẫn Precision**
- Câu có **< 5** id → tự bỏ điểm Recall vô cớ (Recall là metric chính, Precision chỉ tie-break)
- id trùng lặp trong một câu, id không phải string, id không tồn tại trong corpus
- `submission.json` bị nằm trong thư mục con của zip (lỗi rất hay gặp → rớt bài nộp)

In [ ]:
#@title Chạy pipeline trên public test
import time

def full_pipeline(qids, qtext):
    b = retrieve_bm25(qids, qtext)
    d = retrieve_dense(qids, qtext)
    h = rrf(b, d)
    return apply_rerank(qids, qtext, h)   # theo RERANK_MODE

pub_qids = list(PUBLIC.keys())
pub_text = [PUBLIC[q] for q in pub_qids]
t0 = time.time()
pub_preds = full_pipeline(pub_qids, pub_text)
print('%d câu | %.1f phút' % (len(pub_qids), (time.time() - t0) / 60))

In [ ]:
#@title Validate + zip
import json, zipfile, os, collections

def build_submission(preds, all_qids, corpus_ids):
    """Luôn trả về ĐÚNG 5 id/câu. Thiếu thì bù bằng ứng viên tốt nhất còn lại."""
    sub, filler = {}, [d for d in DOC_IDS[:TOPK_SUBMIT]]
    for q in all_qids:
        ids, seen = [], set()
        for d in preds.get(q, []):
            d = str(d)
            if d not in seen and d in corpus_ids:
                seen.add(d); ids.append(d)
            if len(ids) == TOPK_SUBMIT:
                break
        for f in filler:                       # câu không có ứng viên -> vẫn đủ 5
            if len(ids) == TOPK_SUBMIT:
                break
            if f not in seen:
                seen.add(f); ids.append(f)
        sub[q] = {'answer': ids}
    return sub

def validate(sub, all_qids, corpus_ids):
    errs = []
    # scoring.py của BTC: len(ids_preds) != len(ids_truth) -> raise Exception
    # -> lệch số câu làm SẬP TOÀN BỘ bài nộp, không phải chỉ mất điểm 1 câu.
    if len(sub) != len(all_qids):
        errs.append(f'SẬP TOÀN BÀI: {len(sub)} câu nộp vs {len(all_qids)} câu đề')
    if set(sub) != set(all_qids):
        errs.append(f'SẬP TOÀN BÀI: thiếu {len(set(all_qids)-set(sub))} qid, '
                    f'thừa {len(set(sub)-set(all_qids))} qid (len(None) -> TypeError)')
    for q, v in sub.items():
        a = v.get('answer')
        if not isinstance(a, list):
            errs.append(f'{q}: answer không phải list'); continue
        if len(a) == 0:
            errs.append(f'{q}: answer rỗng -> câu này 0 điểm')
        if len(a) > TOPK_SUBMIT:
            errs.append(f'{q}: {len(a)} id > {TOPK_SUBMIT} -> CÂU NÀY BỊ 0 ĐIỂM')
        if len(a) < TOPK_SUBMIT:
            errs.append(f'{q}: chỉ {len(a)} id -> mất điểm Recall vô cớ')
        if len(set(a)) != len(a):
            errs.append(f'{q}: có id trùng lặp')
        if any(not isinstance(x, str) for x in a):
            errs.append(f'{q}: id phải là string')
        if any(x not in corpus_ids for x in a):
            errs.append(f'{q}: có id không tồn tại trong corpus')
    return errs

corpus_ids = set(corpus.doc_id)
sub = build_submission(pub_preds, list(PUBLIC), corpus_ids)
errs = validate(sub, list(PUBLIC), corpus_ids)

if errs:
    print(f'{len(errs)} LỖI — chưa nộp được:')
    for e in errs[:20]:
        print('  -', e)
else:
    # Kaggle: ghi thẳng vào GỐC /kaggle/working -> hiện ngay tab Output, tải 1 click.
    # KHÔNG ghi vào /kaggle/temp: thư mục đó không vào output, session chết là mất file.
    SUB_DIR  = '/kaggle/working' if ON_KAGGLE else ART
    out_json = f'{SUB_DIR}/submission.json'
    out_zip  = f'{SUB_DIR}/submission_{PROFILE}.zip'
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(sub, f, ensure_ascii=False)
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(out_json, arcname='submission.json')   # arcname -> nằm ở GỐC zip
    with zipfile.ZipFile(out_zip) as z:
        assert z.namelist() == ['submission.json'], z.namelist()
    print('OK ->', out_zip, '(%.0f KB)' % (os.path.getsize(out_zip) / 1e3))
    print('     ', out_json)
    if ON_KAGGLE:
        print('Tải về: panel bên phải -> Output -> /kaggle/working -> '
              f'submission_{PROFILE}.zip (hoặc Save Version rồi tải từ tab Output).')
        print('Nộp file ZIP lên Codabench, không nộp file .json.')
    print('phân bố số id/câu:', collections.Counter(len(v['answer']) for v in sub.values()))
    print(json.dumps(dict(list(sub.items())[:2]), ensure_ascii=False, indent=1))

---
## Stage 10 — Xem lại log thí nghiệm

In [ ]:
#@title Bảng thí nghiệm
import pandas as pd, os
if os.path.exists(LOG_CSV):
    df = pd.read_csv(LOG_CSV)
    cols = [c for c in ['time','profile','run','OFFICIAL_recall',
                        'recall@5','recall@20','recall@100','note'] if c in df.columns]
    display(df[cols].tail(25))
else:
    print('chưa có log')

---
## Stage 11 — Lưu artifact để session sau dùng lại (Kaggle)

Chạy trước khi bấm **Save Version**: xem cái gì đang chiếm chỗ trong 20GB output và
kiểm tra embedding đã đủ shard chưa.


In [ ]:
#@title Kiểm kê artifact + hướng dẫn tái dùng
import os, glob

def du(path):
    tot = 0
    for r, _, fs in os.walk(path):
        for f in fs:
            try: tot += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return tot

print('ART =', ART)
rows = []
for p in sorted(glob.glob(f'{ART}/*')) + sorted(glob.glob(f'{ART}/emb/*')):
    if p.rstrip('/').endswith('/emb'):
        continue
    n = len(glob.glob(f'{p}/*')) if os.path.isdir(p) else 1
    rows.append((du(p) if os.path.isdir(p) else os.path.getsize(p), n, p))
for sz, n, p in sorted(rows, reverse=True):
    print(f'  {sz/1e6:9.1f} MB  {n:5d} file  {p.replace(ART + "/", "")}')
print(f'  {du(ART)/1e6:9.1f} MB  TỔNG' + (' / 20000 MB giới hạn Kaggle' if ON_KAGGLE else ''))

if ON_KAGGLE:
    print("""
Để session sau không phải encode lại:
  1. Save Version (Quick Save cũng được — output vẫn được lưu)
  2. Notebook mới / lần chạy sau: Add Input -> Your Work -> chọn notebook này
  3. Stage 0 sẽ tự thấy nó trong ART_IN, Stage 5 chỉ encode shard còn thiếu

Nếu output vượt 20GB, xoá bớt embedding của profile không dùng nữa bằng lệnh:
  (thêm dấu ! ở đầu)  rm -rf {ART}/emb/<tag-cũ>""".replace('{ART}', ART))
else:
    print('\nColab: artifact đã nằm trên Drive, không cần làm gì thêm.')


---
## Việc cần làm ngoài notebook

**1. Đăng ký model với BTC — bắt buộc, hạn 18/09.**
Bài nộp dùng model chưa được phê duyệt sẽ **không được công nhận**.
[Form đăng ký](https://forms.gle/HWE7tcxzWq63Kxv28) ·
[Danh sách đã duyệt](https://docs.google.com/spreadsheets/d/1c5jzsYezWho1WGLRfMKWOaFPLIk_GTnXP5vV8AOWM2Q/edit)

Notebook này dùng: `bkai-foundation-models/vietnamese-bi-encoder` (135M) +
`BAAI/bge-reranker-v2-m3` (568M) = **0.70B < 4B**. Nên đăng ký rộng hơn thực dùng ngay
từ đầu (thêm vài phương án dự phòng) vì chờ duyệt mất thời gian — và nếu bạn chạy cả hai
notebook thì phải đăng ký **cả 4 model**.

**2. Nộp bằng đúng Organization `SGU_LegalMind`** trên Codabench. Bài nộp không qua
Organization là **không hợp lệ**.

**3. Ngân sách 10 lượt nộp/ngày.** Tune hoàn toàn trên val offline, chỉ nộp khi val có cải
thiện thật. 1.000 câu public là tập nhỏ — chênh 1–2% rất có thể chỉ là nhiễu, đừng
overfit leaderboard, vì private test mới là điểm ăn thua.

## Thứ tự chạy đề xuất

| # | Việc | GPU? | Kỳ vọng |
|---|---|---|---|
| 1 | Stage 1–4 (BM25) | **không** | so với TF-IDF `recall@5=0.486` |
| 3 | Stage 5 encode + dense | có | `recall@100` nên > 0.90 |
| 4 | Stage 6 RRF | có | tăng nhẹ so với từng nhánh |
| 5 | Stage 7 rerank → **nộp lần đầu** | có | bước tăng mạnh nhất |
| 6 | Stage 8 fine-tune → encode lại → nộp | có | bước tăng lớn thứ hai |

Chạy bước 2 trong session **CPU** để giữ quota GPU — đó là tài nguyên khan hiếm nhất
trong 38 ngày còn lại.

## Nếu muốn đẩy xa hơn
- **Fine-tune cả reranker** trên train split (thường hơn fine-tune bi-encoder).
- **Ensemble** nhiều bi-encoder (e5, gte-multilingual) rồi RRF — vẫn dư ngân sách 4B.
- **Query expansion**: câu hỏi chỉ ~20 từ, rất ngắn.
- **Doc-level prior**: 3.105/8.532 văn bản từng là gold trong train — phân bố chủ đề của
  corpus có thể khai thác được, nhưng cẩn thận overfit.
